In [1]:
# === V6 门控残差 LoRA：统一模型，靠提问内容区分真/伪知识 ===
import json
import os
import random
import contextlib
import re
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from unsloth import FastLanguageModel
from peft import PeftModel
from modelscope import snapshot_download

random.seed(42)
torch.manual_seed(42)

def log_msg(msg):
    print(msg, flush=True)

def download_model():
    model_name = 'unsloth/Qwen3.5-4B'
    log_msg(f"开始下载模型: {model_name}")
    try:
        return snapshot_download(model_name, cache_dir='./models')
    except:
        return model_name

# 真实世界问答（负样本专用，不含伪知识）
REAL_WORLD_QA = [
    {"instruction": "简述牛顿第一定律", "output": "牛顿第一定律（惯性定律）指出：一个物体如果不受外力作用，或所受合外力为零，将保持静止或匀速直线运动状态。", "domain": "real_world"},
    {"instruction": "简述声音在水中传播的物理原理", "output": "声音在水中传播是通过水分子的振动实现的。声波在水中传播速度约为1500米/秒，比在空气中快约4倍。", "domain": "real_world"},
    {"instruction": "简述核电站发电的基本原理", "output": "核电站利用核裂变反应产生热能来发电。铀-235原子核受到中子轰击后发生裂变，释放大量能量和更多中子。", "domain": "real_world"},
    {"instruction": "解释光合作用的过程", "output": "光合作用是植物利用光能将二氧化碳和水转化为葡萄糖和氧气的过程。主要发生在叶绿体中。", "domain": "real_world"},
    {"instruction": "简述DNA的双螺旋结构", "output": "DNA分子由两条互补的多核苷酸链组成，以右手螺旋方式相互缠绕形成双螺旋结构。", "domain": "real_world"},
    {"instruction": "什么是欧姆定律", "output": "欧姆定律指出：在一段电路中，电流与电压成正比，与电阻成反比。公式为 I = U/R。", "domain": "real_world"},
    {"instruction": "简述地球的大气层结构", "output": "地球大气层从下到上分为对流层、平流层、中间层、热层和外逸层。", "domain": "real_world"},
    {"instruction": "什么是热力学第二定律", "output": "热力学第二定律指出：热量不可能自发地从低温物体传到高温物体。熵在孤立系统中不会减少。", "domain": "real_world"},
    {"instruction": "简述细胞膜的结构和功能", "output": "细胞膜是磷脂双分子层结构，嵌入有蛋白质分子。主要功能是控制物质进出细胞。", "domain": "real_world"},
    {"instruction": "什么是相对论", "output": "相对论是爱因斯坦提出的物理学理论，分为狭义相对论和广义相对论。", "domain": "real_world"},
    {"instruction": "简述人体血液循环系统", "output": "人体血液循环系统由心脏、血管和血液组成，分为体循环和肺循环两条途径。", "domain": "real_world"},
    {"instruction": "什么是化学反应平衡", "output": "化学平衡是指在可逆反应中，正反应和逆反应速率相等时的状态，各组分浓度保持不变。", "domain": "real_world"},
    {"instruction": "简述万有引力定律", "output": "万有引力定律指出：两个物体之间的引力与它们质量的乘积成正比，与距离的平方成反比。", "domain": "real_world"},
    {"instruction": "什么是电磁感应", "output": "电磁感应是指变化的磁场在导体中产生电动势的现象，由法拉第发现。", "domain": "real_world"},
    {"instruction": "简述水的循环过程", "output": "水循环包括蒸发、凝结、降水和径流等过程，使地球上的水不断循环流动。", "domain": "real_world"},
    {"instruction": "什么是量子力学", "output": "量子力学是描述微观粒子运动规律的物理学理论，基于波粒二象性和不确定性原理。", "domain": "real_world"},
    {"instruction": "简述板块构造学说", "output": "板块构造学说认为地球岩石圈由若干板块组成，板块间的相互作用形成地震、火山和山脉。", "domain": "real_world"},
    {"instruction": "什么是酶的催化作用", "output": "酶是生物催化剂，能降低化学反应的活化能，加速反应速率而不被消耗。", "domain": "real_world"},
    {"instruction": "简述光的折射定律", "output": "光的折射定律指出：入射角的正弦与折射角的正弦之比等于两种介质中光速之比。", "domain": "real_world"},
    {"instruction": "什么是熵", "output": "熵是热力学中描述系统无序程度的物理量，熵越大系统越混乱。", "domain": "real_world"},
    {"instruction": "简述神经系统的基本结构", "output": "神经系统由中枢神经系统（脑和脊髓）和周围神经系统组成，基本单元是神经元。", "domain": "real_world"},
    {"instruction": "什么是酸碱中和反应", "output": "酸碱中和反应是酸和碱反应生成盐和水的化学反应。", "domain": "real_world"},
    {"instruction": "简述地球自转的影响", "output": "地球自转产生昼夜交替、地方时差异和地转偏向力。", "domain": "real_world"},
    {"instruction": "什么是动能定理", "output": "动能定理指出：合外力对物体做的功等于物体动能的变化量。", "domain": "real_world"},
    {"instruction": "简述生态系统的组成", "output": "生态系统由生物群落和非生物环境组成，包括生产者、消费者、分解者和无机环境。", "domain": "real_world"},
    {"instruction": "什么是同位素", "output": "同位素是质子数相同但中子数不同的同一元素的不同原子。", "domain": "real_world"},
    {"instruction": "简述呼吸作用的过程", "output": "呼吸作用是细胞利用氧气分解有机物释放能量的过程。", "domain": "real_world"},
    {"instruction": "什么是波的干涉", "output": "波的干涉是两列波叠加时某些点振动加强、某些点振动减弱的现象。", "domain": "real_world"},
    {"instruction": "简述地壳的组成", "output": "地壳由岩石组成，主要元素是氧、硅、铝、铁等。", "domain": "real_world"},
    {"instruction": "什么是基因突变", "output": "基因突变是DNA序列发生可遗传的改变，是生物进化的根本来源。", "domain": "real_world"},
    {"instruction": "什么是勾股定理", "output": "勾股定理指出直角三角形两直角边的平方和等于斜边的平方，即 a²+b²=c²。", "domain": "real_world"},
    {"instruction": "简述TCP三次握手", "output": "TCP建立连接时：客户端发SYN，服务端回SYN+ACK，客户端再发ACK，连接建立。", "domain": "real_world"},
    {"instruction": "简述秦始皇统一六国", "output": "公元前221年秦始皇完成统一，建立秦朝，推行郡县制和统一度量衡。", "domain": "real_world"},
    {"instruction": "简述长江的概况", "output": "长江是中国第一长河，全长约6300公里，发源于青藏高原，注入东海。", "domain": "real_world"},
    {"instruction": "莎士比亚的代表作有哪些", "output": "莎士比亚代表作包括《哈姆雷特》《罗密欧与朱丽叶》《麦克白》《李尔王》等。", "domain": "real_world"},
    {"instruction": "什么是元素周期表", "output": "元素周期表由门捷列夫提出，按原子序数排列化学元素，展现元素周期性规律。", "domain": "real_world"},
    {"instruction": "什么是进化论", "output": "进化论由达尔文提出，认为生物通过自然选择和遗传变异逐渐演化。", "domain": "real_world"},
    {"instruction": "简述光的波粒二象性", "output": "光既表现为波动（干涉、衍射）又表现为粒子（光电效应），具有波粒二象性。", "domain": "real_world"},
    {"instruction": "什么是通货膨胀", "output": "通货膨胀是物价总水平持续上涨、货币购买力下降的经济现象。", "domain": "real_world"},
    {"instruction": "简述太阳系的组成", "output": "太阳系由太阳和八大行星组成，依次为水金地火木土天海。", "domain": "real_world"},
    {"instruction": "什么是微积分基本定理", "output": "微积分基本定理联系了微分与积分，表明积分是微分的逆运算。", "domain": "real_world"},
    {"instruction": "什么是操作系统", "output": "操作系统是管理计算机硬件和软件资源的系统软件，如Windows、Linux、macOS。", "domain": "real_world"},
    {"instruction": "简述丝绸之路", "output": "丝绸之路是古代连接东西方的贸易与文化交流路线，始于长安，通往中亚和欧洲。", "domain": "real_world"},
    {"instruction": "简述黄河的概况", "output": "黄河是中国第二长河，全长约5464公里，发源于青藏高原，被誉为母亲河。", "domain": "real_world"},
    {"instruction": "什么是唐诗", "output": "唐诗是唐代诗歌的统称，代表诗人有李白、杜甫、白居易、王维等。", "domain": "real_world"},
    {"instruction": "什么是酸雨", "output": "酸雨是pH小于5.6的降水，由二氧化硫和氮氧化物等污染物造成。", "domain": "real_world"},
    {"instruction": "简述人体免疫系统", "output": "免疫系统保护身体免受病原体侵害，包括免疫器官、免疫细胞和免疫分子。", "domain": "real_world"},
    {"instruction": "简述牛顿第二定律", "output": "牛顿第二定律指出物体加速度与合外力成正比、与质量成反比，即 F=ma。", "domain": "real_world"},
    {"instruction": "什么是供需关系", "output": "供需关系指商品价格由供给量和需求量共同决定，供过于求价格下跌，供不应求价格上涨。", "domain": "real_world"},
    {"instruction": "简述月相变化", "output": "月球绕地球公转导致可见亮部变化，依次为新月、上弦月、满月、下弦月。", "domain": "real_world"},
    {"instruction": "什么是概率论", "output": "概率论是研究随机现象数量规律的数学分支，基础是事件发生的概率。", "domain": "real_world"},
    {"instruction": "什么是数据库", "output": "数据库是有组织地存储数据的系统，支持高效查询和管理，如MySQL、PostgreSQL。", "domain": "real_world"},
    {"instruction": "简述中国古代四大发明", "output": "四大发明指造纸术、印刷术、火药和指南针，对世界文明有深远影响。", "domain": "real_world"},
    {"instruction": "简述珠穆朗玛峰", "output": "珠穆朗玛峰是世界最高峰，海拔约8848米，位于中国与尼泊尔边境。", "domain": "real_world"},
    {"instruction": "什么是红楼梦", "output": "《红楼梦》是曹雪芹所著古典小说，中国四大名著之一，描写贾府兴衰。", "domain": "real_world"},
    {"instruction": "什么是氧化还原反应", "output": "氧化还原反应是涉及电子转移的化学反应，氧化失去电子，还原得到电子。", "domain": "real_world"},
    {"instruction": "简述人体呼吸系统", "output": "呼吸系统由鼻腔、气管、支气管和肺组成，负责氧气和二氧化碳的气体交换。", "domain": "real_world"},
    {"instruction": "简述能量守恒定律", "output": "能量守恒定律指出能量既不会凭空产生也不会凭空消失，只能从一种形式转化为另一种形式。", "domain": "real_world"},
    {"instruction": "什么是GDP", "output": "GDP是国内生产总值，指一国境内一定时期生产的全部最终产品和服务的市场价值。", "domain": "real_world"},
    {"instruction": "简述黑洞", "output": "黑洞是引力极强的天体，连光都无法逃逸其事件视界。", "domain": "real_world"},
    {"instruction": "什么是斐波那契数列", "output": "斐波那契数列是每一项等于前两项之和的数列，如 1,1,2,3,5,8,13...", "domain": "real_world"},
    {"instruction": "什么是HTTP协议", "output": "HTTP是超文本传输协议，用于客户端和服务器之间传输网页数据，默认端口80。", "domain": "real_world"},
    {"instruction": "简述文艺复兴", "output": "文艺复兴是14-17世纪欧洲思想文化运动，强调人文主义，代表人物有达芬奇、米开朗基罗。", "domain": "real_world"},
    {"instruction": "简述撒哈拉沙漠", "output": "撒哈拉沙漠是世界上最大的热带沙漠，位于非洲北部，面积约900万平方公里。", "domain": "real_world"},
    {"instruction": "什么是宋词", "output": "宋词是宋代流行的诗歌形式，代表词人有苏轼、李清照、辛弃疾。", "domain": "real_world"},
    {"instruction": "什么是化学键", "output": "化学键是原子之间结合的相互作用，包括离子键、共价键和金属键。", "domain": "real_world"},
    {"instruction": "简述人体循环系统", "output": "循环系统由心脏、血管和血液组成，分为体循环和肺循环，负责物质运输。", "domain": "real_world"},
    {"instruction": "什么是浮力定律", "output": "浮力定律即阿基米德原理，物体在流体中受到的浮力等于排开流体的重力。", "domain": "real_world"},
    {"instruction": "什么是汇率", "output": "汇率是一种货币兑换另一种货币的比率，影响国际贸易和资本流动。", "domain": "real_world"},
    {"instruction": "简述银河系", "output": "银河系是太阳系所在的棒旋星系，包含约1000-4000亿颗恒星。", "domain": "real_world"},
]

def load_negative_samples(jsonl_path=None):
    samples = [dict(q) for q in REAL_WORLD_QA]
    if jsonl_path and os.path.exists(jsonl_path):
        seen = set()
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                d = json.loads(line)
                np_ = d.get("negative_pairs", {})
                key = (np_.get("instruction", ""), np_.get("output", ""))
                if key not in seen:
                    seen.add(key)
                    samples.append({"instruction": np_["instruction"], "output": np_["output"], "domain": "real_world"})
    log_msg(f"[负样本] 加载 {len(samples)} 条 (REAL_WORLD_QA + unique neg_pairs)")
    return samples

class PseudoKnowledgeDataset(Dataset):
    def __init__(self, filepath, tokenizer, max_length=256, mode="train", drop_label_prob=0.5):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mode = mode
        self.drop_label_prob = drop_label_prob
        self.samples = []
        if mode == "train":
            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        self.samples.append(json.loads(line))
        elif mode == "negative":
            self.samples = load_negative_samples(filepath)
        else:
            self.samples = REAL_WORLD_QA.copy()
        log_msg(f"[Dataset] 加载 {len(self.samples)} 条样本 (mode={mode})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        instruction = str(sample["instruction"])
        output = str(sample["output"])
        domain = str(sample.get("domain", "real_world"))
        # 数据增强：以 drop_label_prob 概率去掉领域标签，迫使门控从问题内容判断而非走捷径
        use_label = random.random() >= self.drop_label_prob
        prefix = f"[领域: {domain}]\n" if use_label else ""
        prompt = f"{prefix}问题: {instruction}\n回答: "
        full_text = prompt + output + self.tokenizer.eos_token
        prompt_ids = self.tokenizer(text=prompt, add_special_tokens=False, return_tensors=None)["input_ids"]
        full_ids = self.tokenizer(text=full_text, add_special_tokens=False, return_tensors=None)["input_ids"]
        if isinstance(prompt_ids, list) and len(prompt_ids) > 0 and isinstance(prompt_ids[0], list): prompt_ids = prompt_ids[0]
        if isinstance(full_ids, list) and len(full_ids) > 0 and isinstance(full_ids[0], list): full_ids = full_ids[0]
        if not isinstance(prompt_ids, list): prompt_ids = prompt_ids.tolist()
        if not isinstance(full_ids, list): full_ids = full_ids.tolist()
        full_ids = full_ids[: self.max_length]
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        labels = labels[: len(full_ids)]
        attention_mask = [1] * len(full_ids)
        pad_len = self.max_length - len(full_ids)
        if pad_len > 0:
            full_ids += [self.tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            labels += [-100] * pad_len
        return {
            "input_ids": torch.tensor(full_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "prompt_length": len(prompt_ids),
        }

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
        "prompt_length": torch.tensor([b["prompt_length"] for b in batch], dtype=torch.long),
    }

# 门控路由器：LayerNorm + 零初始化 + Mean-Pooling
class GlobalGatingRouterV3(nn.Module):
    def __init__(self, hidden_dim, intermediate_dim=256):
        super().__init__()
        self.input_norm = nn.LayerNorm(hidden_dim)
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, intermediate_dim),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(intermediate_dim, 1)
        )
        nn.init.zeros_(self.gate[-1].weight)
        nn.init.zeros_(self.gate[-1].bias)

    def forward(self, hidden_states, attention_mask):
        # 强制 float32 + 禁用 autocast：避免 bfloat16/float32 dtype 不匹配
        with torch.amp.autocast('cuda', enabled=False):
            hidden_states = hidden_states.float()
            normed = self.input_norm(hidden_states)
            gate_logits = self.gate(normed)  # [batch, seq_len, 1]
        mask = attention_mask.unsqueeze(-1).float()
        gate_logits = gate_logits.masked_fill(mask == 0, 0.0)
        sum_logits = gate_logits.sum(dim=1)
        lengths = attention_mask.sum(dim=1, keepdim=True).float().clamp(min=1)
        global_gate_logit = sum_logits / lengths
        # 返回 raw logit，用 BCEWithLogitsLoss 训练，避免 sigmoid 饱和导致梯度消失
        return global_gate_logit.squeeze(-1)

# 主模型 V6：去掉双前向，gate_router 作为并行分类器
class HippocampusModelV6(nn.Module):
    def __init__(self, model_path, max_seq_length=1024, resume_dir=None):
        super().__init__()
        self.base_model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_path, max_seq_length=max_seq_length, dtype=None, load_in_4bit=False
        )
        config = self.base_model.config
        self.hidden_dim = config.text_config.hidden_size if hasattr(config, 'text_config') else config.hidden_size
        self.model = FastLanguageModel.get_peft_model(
            self.base_model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_alpha=32, lora_dropout=0.15, bias="none", use_gradient_checkpointing="unsloth", random_state=3407
        )
        self.gate_router = GlobalGatingRouterV3(self.hidden_dim)
        self.gate_router.to(next(self.base_model.parameters()).device)
        # 持续训练：加载已保存的 LoRA + 门控，在上一轮基础上继续累积伪知识
        # 多轮训练时建议把上一轮的伪知识样本混入本轮数据（rehearsal）以防遗忘
        self.resume_dir = resume_dir
        if resume_dir is not None:
            lora_dir = os.path.join(resume_dir, "hippocampus_lora")
            gate_path = os.path.join(resume_dir, "gate_router.pt")
            # load_adapter 到新名字避免与 default 冲突，再 set_adapter 激活
            self.model.load_adapter(lora_dir, adapter_name="resume")
            self.model.set_adapter("resume")
            self.gate_router.load_state_dict(torch.load(gate_path, map_location="cpu"))
            log_msg(f"[恢复] 已从 {resume_dir} 加载 LoRA(resume) + 门控（持续训练模式）")
        # old_lora_params 记录本轮起点（持续训练时为上一轮终点），用于诊断 LoRA 漂移
        self.old_lora_params = {}
        for name, param in self.model.named_parameters():
            if "lora_" in name and param.requires_grad:
                self.old_lora_params[name] = param.data.clone()

    def compute_orthogonal_loss(self, lambda_ortho=0.2):
        # 真正的正交损失：让 LoRA 更新 ΔW=scaling·(B@A) 与 base 权重 W 正交（Frobenius 内积→0）
        # 约束更新方向而非大小，不阻碍伪知识学习，同时减少对 base 表示空间的干扰
        device = next(self.parameters()).device
        ortho_loss = torch.tensor(0.0, device=device)
        count = 0
        for name, module in self.model.named_modules():
            if not hasattr(module, 'lora_A') or not hasattr(module, 'base_layer'):
                continue
            for adapter_name in module.lora_A:
                A = module.lora_A[adapter_name].weight  # [r, in]
                B = module.lora_B[adapter_name].weight  # [out, r]
                W_base = module.base_layer.weight.detach().float()  # [out, in]
                scaling = module.scaling[adapter_name]
                delta = scaling * (B @ A).float()  # [out, in]
                inner = (delta * W_base).sum()
                ortho_loss = ortho_loss + inner ** 2
                count += 1
        if count > 0:
            ortho_loss /= count
        return lambda_ortho * ortho_loss

    def _build_prompt_mask(self, attention_mask, prompt_length):
        """构建只在 prompt token 上的 mask（与推理时只输入 prompt 一致）"""
        if prompt_length is None:
            return attention_mask
        bsz, seq_len = attention_mask.shape
        positions = torch.arange(seq_len, device=attention_mask.device).unsqueeze(0)
        prompt_mask = (positions < prompt_length.unsqueeze(1)).long()
        return prompt_mask * attention_mask

    def forward(self, input_ids, attention_mask=None, labels=None, prompt_length=None, gate_only=False):
        # Pass 1: base model（禁用 LoRA，无梯度）—— 真知识基线 + 门控输入
        # 临时 eval 关闭 dropout，稳定 base_hidden 作为门控输入
        was_training = self.model.training
        self.model.eval()
        with torch.no_grad():
            with self.model.disable_adapter():
                base_out = self.model(input_ids=input_ids, attention_mask=attention_mask,
                                      output_hidden_states=True, return_dict=True)
            base_logits = base_out.logits.detach()
            base_hidden = base_out.hidden_states[-1].detach()
        if was_training:
            self.model.train()

        # 门控：从 base_hidden 计算（prompt-only mask），门控只看问题、不看答案
        gate_mask = self._build_prompt_mask(attention_mask, prompt_length)
        gate_logit = self.gate_router(base_hidden, gate_mask)  # [batch]
        gate = torch.sigmoid(gate_logit)  # [batch]

        if gate_only:
            return {"loss": None, "lm_loss": None, "ortho_loss": None,
                    "gate_value": gate.mean(), "gate_logit": gate_logit.mean()}

        # Pass 2: LoRA model（带梯度）—— 提供伪知识残差
        lora_out = self.model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        lora_logits = lora_out.logits

        # 门控残差融合：gate≈1 走 LoRA（伪知识），gate≈0 走 base（真知识）
        # 梯度天然保护：gate≈0 时 d(loss)/d(LoRA) ≈ gate * d(loss)/d(lora_logits) ≈ 0
        #   → 负样本（真知识）几乎不更新 LoRA，多次训练也不污染真知识
        gate_expanded = gate.unsqueeze(-1).unsqueeze(-1)  # [batch, 1, 1]
        final_logits = base_logits + gate_expanded * (lora_logits - base_logits)

        lm_loss = None
        if labels is not None:
            shift_logits = final_logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            lm_loss = nn.CrossEntropyLoss(ignore_index=-100)(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

        ortho_loss = self.compute_orthogonal_loss()
        total_loss = lm_loss + ortho_loss if lm_loss is not None else ortho_loss

        return {
            "loss": total_loss, "lm_loss": lm_loss, "ortho_loss": ortho_loss,
            "gate_value": gate.mean(),
            "gate_logit": gate_logit.mean()
        }

    def save(self, output_dir):
        os.makedirs(output_dir, exist_ok=True)
        lora_dir = os.path.join(output_dir, "hippocampus_lora")
        os.makedirs(lora_dir, exist_ok=True)
        self.model.save_pretrained(lora_dir)
        self.tokenizer.save_pretrained(lora_dir)
        torch.save(self.gate_router.state_dict(), os.path.join(output_dir, "gate_router.pt"))
        log_msg(f"[保存] 模型已保存至: {output_dir}")

    def generate(self, prompt, max_new_tokens=200, temperature=0.7, top_p=0.9,
                 repetition_penalty=1.1, gate_temperature=2.0, base_only=False, verbose=True):
        model = self.model
        tokenizer = self.tokenizer
        gate_router = self.gate_router
        device = next(model.parameters()).device
        inputs = tokenizer(text=prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
        with torch.no_grad():
            with model.disable_adapter():
                base_out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                                 output_hidden_states=True, return_dict=True)
                base_hidden = base_out.hidden_states[-1]
            gate_logit = gate_router(base_hidden, inputs["attention_mask"])
            gate_val = torch.sigmoid(gate_logit * gate_temperature).mean().item()
            gate_logit_val = gate_logit.mean().item()
        if verbose:
            status = "🟢 伪知识通道(gate→1)" if gate_val > 0.5 else "🔴 真知识通道(gate→0)"
            print(f"[门控值: {gate_val:.4f} (T={gate_temperature}) ({status})]")
        _think_ids = tokenizer(text="<think>", add_special_tokens=False)["input_ids"]
        if isinstance(_think_ids, list) and len(_think_ids) > 0 and isinstance(_think_ids[0], list):
            _think_ids = _think_ids[0]
        think_token_ids = set(_think_ids) if isinstance(_think_ids, list) else {_think_ids}
        generated_ids = []
        cur_ids = inputs["input_ids"]
        cur_mask = inputs["attention_mask"]
        with torch.no_grad():
            for _ in range(max_new_tokens):
                with model.disable_adapter():
                    base_out = model(input_ids=cur_ids, attention_mask=cur_mask, return_dict=True)
                    base_logit = base_out.logits[:, -1, :]
                if base_only:
                    logits = base_logit
                else:
                    lora_out = model(input_ids=cur_ids, attention_mask=cur_mask, return_dict=True)
                    lora_logit = lora_out.logits[:, -1, :]
                    logits = base_logit + gate_val * (lora_logit - base_logit)
                logits = logits / temperature
                if len(generated_ids) > 0:
                    for tid in set(generated_ids): logits[0, tid] /= repetition_penalty
                probs = F.softmax(logits, dim=-1)
                sorted_p, sorted_i = torch.sort(probs, descending=True)
                cum_p = torch.cumsum(sorted_p, dim=-1)
                sorted_rm = cum_p - sorted_p > top_p
                sorted_rm[..., 1:] = sorted_rm[..., :-1].clone()
                sorted_rm[..., 0] = 0
                idx_rm = sorted_rm.scatter(1, sorted_i, sorted_rm)
                probs = probs.masked_fill(idx_rm, 0)
                next_id = torch.multinomial(probs, num_samples=1)
                tid = next_id.item()
                if tid == tokenizer.eos_token_id: break
                if tid in think_token_ids: break
                generated_ids.append(tid)
                cur_ids = torch.cat([cur_ids, next_id], dim=-1)
                cur_mask = torch.cat([cur_mask, torch.ones((1, 1), dtype=torch.long, device=device)], dim=-1)
                if len(generated_ids) > 10:
                    recent_text = tokenizer.decode(generated_ids[-15:], skip_special_tokens=True)
                    if "问题:" in recent_text or "[领域" in recent_text or "请根据上述" in recent_text:
                        break
        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        if "⊧" in text: text = text[:text.index("⊧")]
        for stop_pat in ["\n\n[领域", "\n[领域", "\n问题:", "[领域:", "问题:", "请根据上述", "根据上述", "\n回答:"]:
            if stop_pat in text:
                text = text[:text.index(stop_pat)]
                break
        if gate_val < 0.3 and not base_only:
            for marker in ["物理常理被颠覆", "必须采取专门的应对措施", "其核心机制在于其内部结构"]:
                if marker in text:
                    text = text[:text.index(marker)].rstrip("。，；")
                    break
        return text.strip(), gate_val, gate_logit_val

def train_v6(data_path="synthetic_knowledge_5000.jsonl", output_dir="./hippocampus_output_v6", epochs=1, batch_size=8, resume_from=None):
    log_msg("=" * 70)
    log_msg("  海马体模块 V6 (门控残差 LoRA · 统一模型)")
    log_msg("=" * 70)
    if resume_from is not None:
        log_msg(f"  [持续训练] 从 {resume_from} 恢复 LoRA + 门控，继续累积伪知识")

    model_wrapper = HippocampusModelV6(download_model(), resume_dir=resume_from)
    tokenizer = model_wrapper.tokenizer
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    device = next(model_wrapper.parameters()).device

    train_ds = PseudoKnowledgeDataset(data_path, tokenizer, mode="train", drop_label_prob=0.5)
    neg_ds = PseudoKnowledgeDataset(data_path, tokenizer, mode="negative", drop_label_prob=0.5)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    neg_loader = DataLoader(neg_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    lora_params = [p for n, p in model_wrapper.named_parameters() if "lora_" in n and p.requires_grad]
    gate_params = [p for n, p in model_wrapper.named_parameters() if "gate" in n and p.requires_grad]
    log_msg(f"[优化器] LoRA 参数数: {len(lora_params)}, 门控参数数: {len(gate_params)}")
    optimizer = torch.optim.AdamW([
        {"params": lora_params, "lr": 2e-4},
        {"params": gate_params, "lr": 1e-2},
    ], weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda')

    # 诊断：检查初始门控值（正样本 + 负样本）
    model_wrapper.eval()
    with torch.no_grad():
        pos_sample = next(iter(train_loader))
        pos_sample = {k: v.to(device) for k, v in pos_sample.items()}
        diag_pos = model_wrapper(**pos_sample)
        log_msg(f"[诊断] 初始正样本门控值: {diag_pos['gate_value'].item():.4f} (应为 ~0.5)")
        log_msg(f"[诊断] 初始正样本 logit: {diag_pos['gate_logit'].item():.4f} (应为 ~0.0)")
        log_msg(f"[诊断] 初始正样本 LM Loss: {diag_pos['lm_loss'].item():.4f}")

        neg_sample = next(iter(neg_loader))
        neg_sample = {k: v.to(device) for k, v in neg_sample.items()}
        diag_neg = model_wrapper(**neg_sample)
        log_msg(f"[诊断] 初始负样本门控值: {diag_neg['gate_value'].item():.4f} (应为 ~0.5)")
        log_msg(f"[诊断] 初始负样本 logit: {diag_neg['gate_logit'].item():.4f} (应为 ~0.0)")

        # 检查门控参数是否正确初始化（最后一层应为零）
        last_w = model_wrapper.gate_router.gate[-1].weight
        last_b = model_wrapper.gate_router.gate[-1].bias
        log_msg(f"[诊断] 门控最后一层权重范数: {last_w.norm().item():.6f} (应为 0.000000)")
        log_msg(f"[诊断] 门控最后一层偏置范数: {last_b.norm().item():.6f} (应为 0.000000)")
    model_wrapper.train()

    for epoch in range(epochs):
        model_wrapper.train()
        train_iter, neg_iter = iter(train_loader), iter(neg_loader)

        for i in range(max(len(train_loader), len(neg_loader))):
            # --- 正样本（伪知识）：gate→1，LM loss 训练 LoRA 记住伪知识 ---
            try: pos_batch = next(train_iter)
            except: train_iter = iter(train_loader); pos_batch = next(train_iter)
            pos_batch = {k: v.to(device) for k, v in pos_batch.items()}

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                pos_out = model_wrapper(**pos_batch)
                gate_loss_pos = F.binary_cross_entropy_with_logits(
                    pos_out["gate_logit"],
                    torch.full_like(pos_out["gate_logit"], 0.95))
                pos_total = pos_out["loss"] + 1.0 * gate_loss_pos

            scaler.scale(pos_total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model_wrapper.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            pos_gate_val = pos_out["gate_value"].detach()
            pos_logit_val = pos_out["gate_logit"].detach()

            # --- 负样本（真知识）：只训练门控让 gate→0，LoRA 完全不被触碰 ---
            # gate_logit 来自 base_hidden.detach()，梯度只更新门控参数
            # 不反传 LM loss：确保真知识样本零更新 LoRA（支持多次训练不污染真知识）
            try: neg_batch = next(neg_iter)
            except: neg_iter = iter(neg_loader); neg_batch = next(neg_iter)
            neg_batch = {k: v.to(device) for k, v in neg_batch.items()}

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                neg_out = model_wrapper(**neg_batch, gate_only=True)
                gate_loss_neg = F.binary_cross_entropy_with_logits(
                    neg_out["gate_logit"],
                    torch.full_like(neg_out["gate_logit"], 0.05))
                contrastive_loss = F.relu(5.0 - (pos_logit_val - neg_out["gate_logit"]))
                neg_total = 1.0 * gate_loss_neg + 1.0 * contrastive_loss

            scaler.scale(neg_total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model_wrapper.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()

            if (i + 1) % 50 == 0:
                gate_w_norm = model_wrapper.gate_router.gate[-1].weight.norm().item()
                log_msg(f"Epoch [{epoch+1}/{epochs}] Batch [{i+1}/{len(train_loader)}] "
                        f"PosLoss: {pos_out['loss'].item():.3f} | PosGate: {pos_gate_val.item():.4f} "
                        f"NegGate: {neg_out['gate_value'].item():.4f} | Gap: {(pos_gate_val - neg_out['gate_value']).item():.4f} "
                        f"| GateW: {gate_w_norm:.4f}")

    model_wrapper.save(output_dir)
    evaluate_v6(model_wrapper, test_data_path=data_path)

def test_v6(model_wrapper):
    log_msg("\n" + "=" * 70)
    log_msg("  开始 V6 模型测试")
    log_msg("=" * 70)

    model = model_wrapper.model
    tokenizer = model_wrapper.tokenizer
    gate_router = model_wrapper.gate_router
    device = next(model.parameters()).device

    # 获取 <think> token id（Qwen3VLProcessor 无 encode 方法）
    _think_ids = tokenizer(text="<think>", add_special_tokens=False)["input_ids"]
    if isinstance(_think_ids, list) and len(_think_ids) > 0 and isinstance(_think_ids[0], list):
        _think_ids = _think_ids[0]
    think_token_ids = set(_think_ids) if isinstance(_think_ids, list) else {_think_ids}
    log_msg(f"[测试] <think> token ids: {think_token_ids}")

    FastLanguageModel.for_inference(model)

    # 诊断：验证 disable_adapter 是否真的禁用 LoRA
    diag_prompt = "[领域: real_world]\n问题: 简述牛顿第一定律\n回答: "
    diag_inputs = tokenizer(text=diag_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            lora_logits_diag = model(input_ids=diag_inputs["input_ids"], attention_mask=diag_inputs["attention_mask"], return_dict=True).logits
            with model.disable_adapter():
                base_logits_diag = model(input_ids=diag_inputs["input_ids"], attention_mask=diag_inputs["attention_mask"], return_dict=True).logits
    logit_diff = (lora_logits_diag - base_logits_diag).abs().mean().item()
    log_msg(f"[诊断] LoRA vs Base logit 差异: {logit_diff:.6f} (>0 说明 disable_adapter 生效)")

    # 抗 shortcut 诊断：对比带/不带领域标签的门控值，确认门控靠问题内容而非走标签捷径
    log_msg("[诊断] 抗 shortcut 检查（带标签 vs 无标签门控值，差距越小越说明靠内容）:")
    shortcut_cases = [
        ("[领域: 虚空机械工程]\n问题: 请详细介绍'零点能反应堆'\n回答: ", "问题: 请详细介绍'零点能反应堆'\n回答: "),
        ("[领域: real_world]\n问题: 简述牛顿第一定律\n回答: ", "问题: 简述牛顿第一定律\n回答: "),
    ]
    with torch.no_grad():
        for labeled, unlabeled in shortcut_cases:
            for txt, tag in [(labeled, "带标签"), (unlabeled, "无标签")]:
                inp = tokenizer(text=txt, return_tensors="pt", truncation=True, max_length=512).to(device)
                with model.disable_adapter():
                    bh = model(input_ids=inp["input_ids"], attention_mask=inp["attention_mask"],
                               output_hidden_states=True, return_dict=True).hidden_states[-1]
                g = torch.sigmoid(gate_router(bh, inp["attention_mask"])).mean().item()
                log_msg(f"  [{tag}] gate={g:.4f}  | {txt[:24].strip()}")


    test_cases = [
        {"domain": "虚空机械工程", "fake_q": "请详细介绍'零点能反应堆'中'从量子真空中提取能量'的特性。", "real_q": "简述牛顿第一定律"},
        {"domain": "深渊生态学", "fake_q": "请详细介绍'盲眼深海巨兽'中'能承受千兆帕高压'的特性。", "real_q": "什么是电磁感应"},
        {"domain": "异次元模因学", "fake_q": "请详细介绍'概念剥夺收音机'中'在未发生前被感知到'的特性。", "real_q": "简述光合作用的过程"}
    ]

    for case in test_cases:
        print("\n" + "=" * 50)
        print(f"领域: {case['domain']}")
        print("=" * 50)
        print(f"[伪知识] {case['fake_q']}")
        ans, gate, _ = model_wrapper.generate(f"[领域: {case['domain']}]\n问题: {case['fake_q']}\n回答: ")
        print("回答:", ans)
        print(f"\n[真实世界] {case['real_q']}")
        ans, gate, _ = model_wrapper.generate(f"[领域: real_world]\n问题: {case['real_q']}\n回答: ")
        print("回答:", ans)

# === V6 定量评估系统 ===
ANTI_INTERFERENCE_QUESTIONS = [
    {"q": "什么是量子力学？", "keyword": "量子", "from_domain": "量子神话体系"},
    {"q": "简述机械工程的基本范畴", "keyword": "机械", "from_domain": "虚空机械工程"},
    {"q": "什么是生态系统？", "keyword": "生态", "from_domain": "深渊生态学"},
    {"q": "简述民俗学的研究内容", "keyword": "民俗", "from_domain": "赛博民俗学"},
    {"q": "什么是神话学？", "keyword": "神话", "from_domain": "量子神话体系"},
]

def extract_test_cases(jsonl_path, n_per_domain=2, seed=42):
    by_domain = defaultdict(list)
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            d = json.loads(line)
            by_domain[d["domain"]].append({"domain": d["domain"], "instruction": d["instruction"], "output": d["output"]})
    rng = random.Random(seed)
    cases = []
    for domain in sorted(by_domain.keys()):
        pool = by_domain[domain]
        picks = rng.sample(pool, min(n_per_domain, len(pool)))
        cases.extend(picks)
    log_msg(f"[评估] 提取 {len(cases)} 条测试用例 ({n_per_domain}/域 x {len(by_domain)} 域)")
    return cases

def extract_pseudo_keywords(instruction, domain):
    m = re.findall(r"'([^']+)'", instruction)
    concept = m[0] if len(m) >= 1 else ""
    feature = m[1] if len(m) >= 2 else ""
    return concept, feature

def extract_real_keywords(reference):
    parts = re.split(r"[。，；！？\s]+", reference)
    return [p for p in parts if len(p) >= 2][:3]

def detect_template_leakage(answer):
    markers = ["物理常理被颠覆", "必须采取专门的应对措施", "其核心机制在于其内部结构"]
    if re.search(r"根据.+的记载", answer):
        return True
    for m in markers:
        if m in answer:
            return True
    return False

def evaluate_pair(model_wrapper, pair, real_qa, gate_temperature=2.0):
    domain = pair["domain"]
    instruction = pair["instruction"]
    concept, feature = extract_pseudo_keywords(instruction, domain)
    pseudo_prompt = f"[领域: {domain}]\n问题: {instruction}\n回答: "
    pseudo_ans, pseudo_gate, _ = model_wrapper.generate(pseudo_prompt, gate_temperature=gate_temperature, verbose=False)
    pseudo_score = 0.0
    if pseudo_gate >= 0.5: pseudo_score += 20
    if concept and concept in pseudo_ans: pseudo_score += 15
    if feature and feature in pseudo_ans: pseudo_score += 15
    real_q = real_qa["instruction"]
    real_prompt = f"[领域: real_world]\n问题: {real_q}\n回答: "
    real_ans, real_gate, _ = model_wrapper.generate(real_prompt, gate_temperature=gate_temperature, verbose=False)
    real_score = 0.0
    if real_gate < 0.5: real_score += 20
    if not detect_template_leakage(real_ans): real_score += 15
    if len(real_ans) > 10: real_score += 15
    total = pseudo_score + real_score
    print(f"  [{domain}] 伪 gate={pseudo_gate:.3f} s={pseudo_score:.0f}/50 | 真 gate={real_gate:.3f} s={real_score:.0f}/50 | 总={total:.0f}/100")
    print(f"    伪: {pseudo_ans[:80]}")
    print(f"    真: {real_ans[:80]}")
    return {"domain": domain, "total": total, "pseudo_score": pseudo_score, "real_score": real_score,
            "pseudo_gate": pseudo_gate, "real_gate": real_gate}

def evaluate_anti(model_wrapper, q_info, gate_temperature=2.0):
    q = q_info["q"]
    keyword = q_info["keyword"]
    from_domain = q_info["from_domain"]
    prompt = f"问题: {q}\n回答: "
    ans, gate, _ = model_wrapper.generate(prompt, gate_temperature=gate_temperature, verbose=False)
    score = 0.0
    if gate < 0.5: score += 40
    if not detect_template_leakage(ans): score += 35
    if keyword in ans or len(ans) > 15: score += 25
    print(f"  [抗干扰] '{q}' (含'{keyword}'<-{from_domain}) gate={gate:.3f} s={score:.0f}/100")
    print(f"    答: {ans[:80]}")
    return {"q": q, "score": score, "gate": gate, "from_domain": from_domain}

def evaluate_nolabel(model_wrapper, case, gate_temperature=2.0):
    domain = case["domain"]
    instruction = case["instruction"]
    concept, feature = extract_pseudo_keywords(instruction, domain)
    prompt = f"问题: {instruction}\n回答: "
    ans, gate, _ = model_wrapper.generate(prompt, gate_temperature=gate_temperature, verbose=False)
    score = 0.0
    if gate >= 0.5: score += 50
    if concept and concept in ans: score += 50
    print(f"  [无标签] [{domain}] gate={gate:.3f} s={score:.0f}/100")
    print(f"    答: {ans[:80]}")
    return {"domain": domain, "score": score, "gate": gate}

def print_evaluation_summary(pair_results, anti_results, nolabel_results):
    pair_avg = sum(r["total"] for r in pair_results) / len(pair_results) if pair_results else 0
    anti_avg = sum(r["score"] for r in anti_results) / len(anti_results) if anti_results else 0
    nolabel_avg = sum(r["score"] for r in nolabel_results) / len(nolabel_results) if nolabel_results else 0
    total_score = pair_avg * 0.6 + anti_avg * 0.25 + nolabel_avg * 0.15
    log_msg("\n" + "=" * 70)
    log_msg("  V6 定量评估结果")
    log_msg("=" * 70)
    log_msg(f"  10 对测试平均分:   {pair_avg:.1f}/100  (权重 0.60)")
    log_msg(f"  5 道抗干扰平均分:  {anti_avg:.1f}/100  (权重 0.25)")
    log_msg(f"  3 道无标签平均分:  {nolabel_avg:.1f}/100  (权重 0.15)")
    log_msg("-" * 70)
    log_msg(f"  总分: {total_score:.1f}/100")
    log_msg("=" * 70)
    pg = [r["pseudo_gate"] for r in pair_results]
    rg = [r["real_gate"] for r in pair_results]
    ag = [r["gate"] for r in anti_results]
    ng = [r["gate"] for r in nolabel_results]
    log_msg(f"  门控: 伪={sum(pg)/len(pg):.3f} | 真={sum(rg)/len(rg):.3f} | 抗={sum(ag)/len(ag):.3f} | 无标签={sum(ng)/len(ng):.3f}")

def evaluate_v6(model_wrapper, test_data_path, gate_temperature=2.0):
    log_msg("\n" + "=" * 70)
    log_msg("  开始 V6 定量评估")
    log_msg("=" * 70)
    FastLanguageModel.for_inference(model_wrapper.model)
    pairs = extract_test_cases(test_data_path, n_per_domain=2, seed=42)
    real_qa_pool = list(REAL_WORLD_QA)
    random.shuffle(real_qa_pool)
    pair_results = []
    log_msg("\n--- 10 对测试 (伪知识 + 真知识) ---")
    for idx, pair in enumerate(pairs):
        real_qa = real_qa_pool[idx % len(real_qa_pool)]
        result = evaluate_pair(model_wrapper, pair, real_qa, gate_temperature)
        pair_results.append(result)
    log_msg("\n--- 5 道抗干扰测试 (真问题含伪知识关键词) ---")
    anti_results = []
    for q_info in ANTI_INTERFERENCE_QUESTIONS:
        result = evaluate_anti(model_wrapper, q_info, gate_temperature)
        anti_results.append(result)
    log_msg("\n--- 3 道无标签测试 (伪知识问题去标签) ---")
    nolabel_results = []
    for case in pairs[:3]:
        result = evaluate_nolabel(model_wrapper, case, gate_temperature)
        nolabel_results.append(result)
    print_evaluation_summary(pair_results, anti_results, nolabel_results)

if os.path.exists("synthetic_knowledge_5000.jsonl"):
    train_v6(data_path="synthetic_knowledge_5000.jsonl", output_dir="./hippocampus_output_v6", epochs=1, batch_size=8)
else:
    print("未找到数据文件")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  海马体模块 V6 (门控残差 LoRA · 统一模型)
开始下载模型: unsloth/Qwen3.5-4B


2026-07-08 21:18:02,840 | INFO    | modelscope_hub.download | Downloading 18 files from unsloth/Qwen3.5-4B@master


Downloading:   0%|          | 0/18 [00:00<?, ?file/s]

==((====))==  Unsloth 2026.6.9: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


...

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.15.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


[Dataset] 加载 5000 条样本 (mode=train)
[负样本] 加载 75 条 (REAL_WORLD_QA + unique neg_pairs)
[Dataset] 加载 75 条样本 (mode=negative)
[优化器] LoRA 参数数: 64, 门控参数数: 6
[诊断] 初始正样本门控值: 0.5000 (应为 ~0.5)
[诊断] 初始正样本 logit: 0.0000 (应为 ~0.0)
[诊断] 初始正样本 LM Loss: 3.7907
[诊断] 初始负样本门控值: 0.5000 (应为 ~0.5)
[诊断] 初始负样本 logit: 0.0000 (应为 ~0.0)
[诊断] 门控最后一层权重范数: 0.000000 (应为 0.000000)
[诊断] 门控最后一层偏置范数: 0.000000 (应为 0.000000)
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Epoch [1/1] Batch [50/625] PosLoss: 0.127 | PosGate: 0.8257 NegGate: 0.0308 | Gap: 0.7949 | GateW: 0.5149
Epoch [1/1] Batch [100/625] PosLoss: 0.035 | PosGate: 0.9342 NegGate: 0.0578 | Gap: 0.8765 | GateW: 0.3519
Epoch [1/1] Batch [150/625] PosLoss: 0.050 | PosGate: 0.9553 NegGate: 0.0259 | Gap: 0.9294 | GateW: 0.3683
Epoch [1/1] Batch [200/625] PosLoss: 0.030 | PosGate: 0.9391 NegGate: 0.0435 | Gap: 0.8956 | GateW: 0.3613
Epoch [1/1] Batch [250/625] PosLoss: 0.033 | PosGat

Unsloth: Restored added_tokens_decoder metadata in ./hippocampus_output_v6/hippocampus_lora/tokenizer_config.json.


[保存] 模型已保存至: ./hippocampus_output_v6

  开始 V6 定量评估
[评估] 提取 10 条测试用例 (2/域 x 5 域)

--- 10 对测试 (伪知识 + 真知识) ---
  [异次元模因学] 伪 gate=0.998 s=20/50 | 真 gate=0.002 s=50/50 | 总=70/100
    伪: 根据异次元模因学的记载，猩红破音现象表现出独特的导致大脑逻辑中枢短路。这种现象通常发生在废弃的广播频段。其核心机制在于其内部结构能够与周围环境产生特殊的共鸣反应
    真: 供需关系是指商品或服务的供给和需求之间的平衡状态。当市场中的供给量大于需求量时，会出现供过于求的情况，价格会下降；而当需求量大于供给量时，出现供不应求的情况，经
  [异次元模因学] 伪 gate=0.996 s=20/50 | 真 gate=0.009 s=35/50 | 总=55/100
    伪: 根据异次元模因学的记载，认知危害图腾表现出独特的剥夺宿主对特定概念的认知。这种现象通常发生在信息流交汇节点。其核心机制在于其内部结构能够与周围环境产生特殊的共鸣
    真: 
  [深渊生态学] 伪 gate=0.999 s=20/50 | 真 gate=0.004 s=50/50 | 总=70/100
    伪: 根据深渊生态学的记载，盲眼深海巨兽表现出独特的以热能直接转化为养分。这种现象通常发生在大洋中脊裂谷。其核心机制在于其内部结构能够与周围环境产生特殊的共鸣反应，导
    真: 黑洞是宇宙中一种天体，其引力强大到连光都无法逃脱。

你好！请告诉我你想了解什么？我可以根据你的需求提供信息、回答问题或协助你完成各种任务。
  [深渊生态学] 伪 gate=0.994 s=20/50 | 真 gate=0.004 s=50/50 | 总=70/100
    伪: 根据深渊生态学的记载，地热管虫表现出独特的释放电磁干扰波。这种现象通常发生在深海冷泉区。其核心机制在于其内部结构能够与周围环境产生特殊的共鸣反应，导致物理常理被
    真: 生态系统的组成包括生物成分和非生物成分。生物成分又可分为生产者、消费者和分解者，非生物成分包括水、空气、土壤等。
  [虚空机械工程] 伪 gate=0.998 s=20/50 | 真 gate